# Monotonic Stack

A stack that maintains elements in sorted order (increasing or decreasing). When pushing a new element, pop all elements that violate the monotonic property.

**Key insight:** Each element is pushed and popped at most once → O(n) total despite the inner while loop.

**When to use:** Problems asking for the **next greater/smaller element**, or **nearest larger/smaller** to the left or right.

| Pattern | Stack type | Pop when |
|---------|-----------|----------|
| Next greater element | Decreasing | `stack[-1] < current` |
| Next smaller element | Increasing | `stack[-1] > current` |

**Time:** O(n) &nbsp; **Space:** O(n)

# Next Greater Element

For each element, find the first element to its **right** that is greater; -1 if there is
none.

Scanning right to left, the stack holds exactly the elements that could still be somebody's
answer. When a new element arrives, everything on the stack smaller than or equal to it is
now useless -- any element further left would meet the new, larger element first -- so those
get popped and thrown away for good. Whatever survives on top is the answer.

```
arr = [4, 5, 2, 25]        stack holds indices; shown as values

i=3 (25)  stack empty            → result[3] = -1     push 25   [25]
i=2 (2)   top 25 > 2             → result[2] = 25     push 2    [25, 2]
i=1 (5)   top 2 <= 5 → pop
          top 25 > 5             → result[1] = 25     push 5    [25, 5]
i=0 (4)   top 5 > 4              → result[0] = 5      push 4    [25, 5, 4]

result: [5, 25, 25, -1]
```

The inner `while` looks like it should make this quadratic, but every element is pushed once
and popped at most once, so the total pop count is bounded by n -- an amortized argument, the
same shape as the dynamic array analysis.

**Time:** O(n) &nbsp; **Space:** O(n)

In [ ]:
def next_greater(arr):
    """For each element, find next greater to the right. Time: O(n)"""
    n = len(arr)
    result = [-1] * n
    stack = []  # stores indices, values are decreasing
    for i in range(n - 1, -1, -1):
        while stack and arr[stack[-1]] <= arr[i]:
            stack.pop()  # smaller elements can't be answer
        if stack:
            result[i] = arr[stack[-1]]
        stack.append(i)
    return result

def test_next_greater():
    assert next_greater([4, 5, 2, 25]) == [5, 25, 25, -1]
    assert next_greater([13, 7, 6, 12]) == [-1, 12, 12, -1]
    assert next_greater([3, 2, 1]) == [-1, -1, -1]

test_next_greater()

# Next Smaller Element

The mirror image, and the code differs by one character: pop while the top is `>=` the
current element instead of `<=`. The stack now keeps *increasing* values from the top down.

That symmetry is the useful takeaway -- "nearest larger" and "nearest smaller" are the same
algorithm with the comparison flipped, and scanning left-to-right instead of right-to-left
flips "next" to "previous". Four problems, one template.

**Time:** O(n) &nbsp; **Space:** O(n)

In [ ]:
def next_smaller(arr):
    """For each element, find next smaller to the right. Time: O(n)"""
    n = len(arr)
    result = [-1] * n
    stack = []  # stores indices, values are increasing
    for i in range(n - 1, -1, -1):
        while stack and arr[stack[-1]] >= arr[i]:
            stack.pop()
        if stack:
            result[i] = arr[stack[-1]]
        stack.append(i)
    return result

def test_next_smaller():
    assert next_smaller([4, 8, 5, 2, 25]) == [2, 5, 2, -1, -1]
    assert next_smaller([1, 2, 3]) == [-1, -1, -1]

test_next_smaller()

# Application: Daily Temperatures

Same question as next-greater, but the answer is the **distance** rather than the value.

That one change makes it natural to flip the scan direction. Going left to right, a
just-arrived element is the answer for everything smaller still on the stack, and because
the stack holds *indices*, the distance falls out as `i - j`. Anything left on the stack at
the end never found a warmer day, and keeps its initial 0.

```
temps = [73, 74, 75, 71, 69, 72, 76, 73]

i=1 (74)  74 > 73 → pop 0, result[0] = 1 - 0 = 1
i=2 (75)  75 > 74 → pop 1, result[1] = 1
i=3 (71)  push
i=4 (69)  push
i=5 (72)  72 > 69 → pop 4, result[4] = 1
          72 > 71 → pop 3, result[3] = 2
i=6 (76)  76 > 72 → pop 5, result[5] = 1
          76 > 75 → pop 2, result[2] = 6 - 2 = 4
i=7 (73)  push       indices 6 and 7 never pop → 0

result: [1, 1, 4, 2, 1, 1, 0, 0]
```

**Time:** O(n) &nbsp; **Space:** O(n)

In [ ]:
def daily_temperatures(temps):
    """Days until warmer temperature. Time: O(n)"""
    n = len(temps)
    result = [0] * n
    stack = []  # decreasing stack of indices
    for i in range(n):
        while stack and temps[i] > temps[stack[-1]]:
            j = stack.pop()
            result[j] = i - j
        stack.append(i)
    return result

def test_daily_temps():
    assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
    assert daily_temperatures([30, 40, 50, 60]) == [1, 1, 1, 0]
    assert daily_temperatures([30, 20, 10]) == [0, 0, 0]

test_daily_temps()

# Application: Largest Rectangle in Histogram

Every candidate rectangle is limited by its shortest bar, so ask a different question: for
each bar, **how wide a rectangle can it support at its own height?** It extends until it
meets a shorter bar on either side, so what is needed is the nearest shorter bar left and
right -- monotonic stack territory.

The stack keeps increasing heights. A bar shorter than the top means the top's right
boundary has been found, so pop it and settle its rectangle: the height is the popped bar,
and the width runs between the new stack top (its left boundary) and the current index.

```
heights = [2, 1, 5, 6, 2, 3] + [0]      sentinel appended

i=1 h=1   pop 2  → width 1, area 2
i=4 h=2   pop 6  → width 4-2-1 = 1, area 6
          pop 5  → width 4-1-1 = 2, area 10   ← best
i=6 h=0   pop 3  → width 1, area 3
          pop 2  → width 6-1-1 = 4, area 8
          pop 1  → stack empty → width 6, area 6

max area: 10   (height 5 spanning bars 5 and 6)
```

Two details that are easy to get wrong:

- The appended `0` sentinel guarantees every remaining bar gets popped and measured; without
  it, an increasing histogram like `[1, 2, 3]` would end with the stack full and the answer
  never computed
- `width = i if not stack else i - stack[-1] - 1` -- an empty stack means the popped bar was
  the shortest so far, so it extends all the way back to index 0

**Time:** O(n) &nbsp; **Space:** O(n)

In [ ]:
def largest_rectangle(heights):
    """Largest rectangle area in histogram. Time: O(n)"""
    stack = []  # increasing stack of indices
    max_area = 0
    heights = heights + [0]  # sentinel to flush remaining bars
    for i, h in enumerate(heights):
        while stack and heights[stack[-1]] > h:
            height = heights[stack.pop()]
            width = i if not stack else i - stack[-1] - 1
            max_area = max(max_area, height * width)
        stack.append(i)
    return max_area

def test_largest_rect():
    assert largest_rectangle([2, 1, 5, 6, 2, 3]) == 10  # 5×2
    assert largest_rectangle([2, 4]) == 4
    assert largest_rectangle([6, 2, 5, 4, 5, 1, 6]) == 12  # 4×3

test_largest_rect()